In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Fully Connected Model Definition
class FullyConnectedModel(nn.Module):
    def __init__(self):
        super(FullyConnectedModel, self).__init__()
        self.fc1 = nn.Linear(128*128*3, 2816)
        self.fc2 = nn.Linear(2816, 2560)
        self.fc3 = nn.Linear(2560, 2304)
        self.fc4 = nn.Linear(2304, 2048)
        self.fc5 = nn.Linear(2048, 1792)
        self.fc6 = nn.Linear(1792, 1536)
        self.fc7 = nn.Linear(1536, 1280)
        self.fc8 = nn.Linear(1280, 1024)
        self.fc9 = nn.Linear(1024, 768)
        self.fc10 = nn.Linear(768, 512)
        self.fc11 = nn.Linear(512, 256)
        self.fc12 = nn.Linear(256, 128)
        self.fc13 = nn.Linear(128, 64)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = torch.relu(self.fc5(x))
        x = torch.relu(self.fc6(x))
        x = torch.relu(self.fc7(x))
        x = torch.relu(self.fc8(x))
        x = torch.relu(self.fc9(x))
        x = torch.relu(self.fc10(x))
        x = torch.relu(self.fc11(x))
        x = torch.relu(self.fc12(x))
        x = self.fc13(x)
        return x

# Convolutional Neural Network (CNN) Definition
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop1 = nn.Dropout(0.25)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop2 = nn.Dropout(0.25)

        self.conv3 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop3 = nn.Dropout(0.25)

        self.conv4 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)

        self.fc1 = nn.Linear(64 * 16 * 16, 64)  # Assuming input size is 128x128
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.drop1(x)

        x = self.pool2(torch.relu(self.conv2(x)))
        x = self.drop2(x)

        x = self.pool3(torch.relu(self.conv3(x)))
        x = self.drop3(x)

        x = torch.relu(self.conv4(x))

        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return x

# Hybrid Model Definition
class HybridModel(nn.Module):
    def __init__(self):
        super(HybridModel, self).__init__()
        self.fc_model = FullyConnectedModel()
        self.cnn_model = CNNModel()
        self.fc_final = nn.Linear(128, 10)

    def forward(self, visual_input, trajectory_input):
        visual_output = self.cnn_model(visual_input)
        trajectory_output = self.fc_model(trajectory_input)

        combined = torch.cat((visual_output, trajectory_output), dim=1)
        output = self.fc_final(combined)
        return output

In [ ]:
import os
from PIL import Image
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Custom Dataset Class
class CustomDataset(Dataset):
    def __init__(self, img_dir, weights_dir, traj_joint_dir, traj_task_dir, transform=None):
        self.img_dir = img_dir
        self.weights_dir = weights_dir
        self.traj_joint_dir = traj_joint_dir
        self.traj_task_dir = traj_task_dir
        self.transform = transform

        self.img_files = sorted(os.listdir(img_dir))
        self.weight_files = sorted(os.listdir(weights_dir))
        self.traj_joint_files = sorted(os.listdir(traj_joint_dir))
        self.traj_task_files = sorted(os.listdir(traj_task_dir))

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_files[idx])
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)

        weight_path = os.path.join(self.weights_dir, self.weight_files[idx])
        weights = np.load(weight_path)
        weights = torch.tensor(weights, dtype=torch.float32)

        traj_joint_path = os.path.join(self.traj_joint_dir, self.traj_joint_files[idx])
        traj_joint = np.load(traj_joint_path)
        traj_joint = torch.tensor(traj_joint, dtype=torch.float32)

        traj_task_path = os.path.join(self.traj_task_dir, self.traj_task_files[idx])
        traj_task = np.load(traj_task_path)
        traj_task = torch.tensor(traj_task, dtype=torch.float32)

        # Assuming the model expects the concatenation of both trajectory data
        traj_data = torch.cat((traj_joint, traj_task), dim=0)

        return img, traj_data, weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Specify the path to the folder in your Google Drive
#drive_folder_path = '/content/drive/MyDrive/Data'
base_path = '/content/drive/MyDrive/Data'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
img_dir = os.path.join(base_path, 'img_encoded')
weights_dir = os.path.join(base_path, 'weights')
traj_joint_dir = os.path.join(base_path, 'traj_joint')
traj_task_dir = os.path.join(base_path, 'traj_task')


# Hyperparameters
num_epochs = 5
batch_size = 32
learning_rate = 0.001

# Define any image transformations
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

# Initialize the custom dataset with the actual paths
#dataset = CustomDataset(
 #   img_dir='/content/drive/MyDrive/Data/img_encoded/',
  #  weights_dir='/content/drive/MyDrive/Data/weights/',
   # traj_joint_dir='/content/drive/MyDrive/Data/traj_joint/',
    #traj_task_dir='/content/drive/MyDrive/Data/traj_task/',
    #transform=transform
#)

dataset = CustomDataset(
    img_dir=img_dir,
    weights_dir=weights_dir,
    traj_joint_dir=traj_joint_dir,
    traj_task_dir=traj_task_dir,
    transform=transform
)


# DataLoader
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize the model, criterion, and optimizer
model = HybridModel()
criterion = nn.MSELoss()  # Using MSE loss for trajectory prediction
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
total_step = len(train_loader)
for epoch in range(num_epochs):
    for i, (visuals, trajectories, targets) in enumerate(train_loader):
        # Forward pass
        outputs = model(visuals, trajectories)
        loss = criterion(outputs, targets)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i+1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{total_step}], Loss: {loss.item():.4f}')

print('Finished Training')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Data/img_encoded'